# umap_learn

## Example 1: embed separated clusters in 2-D

In [ ]:
# Example 1: UMAP keeps three well-separated blobs apart
import numpy as np
import umap
from sklearn.datasets import make_blobs

data, labels = make_blobs(n_samples=300, centers=3, n_features=10, random_state=0)
embedding = umap.UMAP(n_neighbors=15, random_state=0).fit_transform(data)
centres = np.array([embedding[labels == k].mean(axis=0) for k in range(3)])
nearest = np.argmin(((embedding[:, None, :] - centres[None]) ** 2).sum(-1), axis=1)
accuracy = float((nearest == labels).mean())
assert embedding.shape == (300, 2) and accuracy > 0.98, accuracy  # every blob stays its own cluster
print("Example 1:", embedding.shape, "nearest-centre accuracy", accuracy)

## Example 2: transform new points into an existing embedding

In [ ]:
# Example 2: a fitted UMAP places unseen points near their cluster
import numpy as np
import umap
from sklearn.datasets import make_blobs

data, labels = make_blobs(n_samples=320, centers=2, n_features=6, random_state=1)
reducer = umap.UMAP(random_state=1).fit(data[:300])
new = reducer.transform(data[300:])
centres = [reducer.embedding_[labels[:300] == k].mean(axis=0) for k in range(2)]
nearest = [int(np.argmin([np.linalg.norm(p - c) for c in centres])) for p in new]
assert np.mean(np.array(nearest) == labels[300:]) >= 0.9, (nearest, labels[300:])
print("Example 2:", new.shape)

## Example 3: supervised UMAP and other metrics

In [ ]:
# Example 3: fit with labels (y=) and a cosine metric
import numpy as np
import umap
from sklearn.datasets import load_iris

iris = load_iris()
embedding = umap.UMAP(metric="cosine", n_components=3, random_state=2).fit_transform(iris.data, y=iris.target)
assert embedding.shape == (150, 3) and np.isfinite(embedding).all()
print("Example 3:", embedding.shape)

## Example 4: the fuzzy simplicial set graph

In [ ]:
# Example 4: the fitted model exposes its sparse neighbour graph (graph_), symmetric with weights in (0, 1]
import umap
from sklearn.datasets import make_blobs

data, _ = make_blobs(n_samples=200, centers=4, n_features=5, random_state=3)
graph = umap.UMAP(n_neighbors=10, random_state=3).fit(data).graph_
assert graph.shape == (200, 200) and abs(graph - graph.T).max() < 1e-6
assert 0 < graph.data.min() and graph.data.max() <= 1.0
print("Example 4:", graph.shape, graph.nnz, "edges")

## Example 5: inverse transform back to data space

In [ ]:
# Example 5: inverse_transform maps embedding points back to the original feature space
import numpy as np
import umap
from sklearn.datasets import make_blobs

data, _ = make_blobs(n_samples=200, centers=2, n_features=4, random_state=4)
reducer = umap.UMAP(random_state=4).fit(data)
back = reducer.inverse_transform(reducer.embedding_[:5])
assert back.shape == (5, 4) and np.isfinite(back).all()
print("Example 5:", np.round(back[0], 2))